# Celda 0: Preparar el entorno
#### Configurar notebook para que encuentre los scripts del trabajo

In [1]:
# Recargar automaticamente los scripts de src/
%reload_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Verificar que estamos ejecutando desde notebooks/
SRC = Path("../src").resolve()

if not SRC.is_dir():
    raise FileNotFoundError(
        "No se encontro ../src/. "
        "Abre el notebook desde la carpeta notebooks/."
    )

# Permitir importar scripts de src/
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

# Definir rutas del proyecto
DATA = Path("../data")
PROC = DATA / "processed"
RAW = DATA / "raw"

RESULTS = Path("../results")
TABLES = RESULTS / "tables"
FIGURES = RESULTS / "figures"

# Crear carpetas de salida
PROC.mkdir(parents=True, exist_ok=True)
TABLES.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

print("Entorno preparado correctamente")
print("Scripts:", SRC)
print("Datos procesados:", PROC.resolve())
print("Tablas:", TABLES.resolve())
print("Figuras:", FIGURES.resolve())

Entorno preparado correctamente
Scripts: C:\Users\omarrm\OneDrive\Documentos\biopython\Plant_pathogen_interaction\src
Datos procesados: C:\Users\omarrm\OneDrive\Documentos\biopython\Plant_pathogen_interaction\data\processed
Tablas: C:\Users\omarrm\OneDrive\Documentos\biopython\Plant_pathogen_interaction\results\tables
Figuras: C:\Users\omarrm\OneDrive\Documentos\biopython\Plant_pathogen_interaction\results\figures


# Celda 1: Obtención y procesamiento de datos KEGG
#### Consultar las asociaciones gen-ruta y normalizar identificadores

In [2]:
from kegg_analysis import fetch_kegg_associations
from kegg_analysis import normalize_kegg_associations

# Definir archivo de salida
kegg_file = PROC / "kegg_gene_pathway.tsv"

# Cargar asociaciones existentes u obtenerlas de KEGG
if kegg_file.exists():
    print("Cargando asociaciones KEGG existentes...")
    kegg_associations_df = pd.read_csv(kegg_file, sep="\t")
else:
    print("Consultando asociaciones en KEGG...")
    kegg_associations_df = fetch_kegg_associations(organism_code="pst")
    kegg_associations_df.to_csv(kegg_file, sep="\t", index=False)

# Normalizar identificadores
normalized_kegg_df = normalize_kegg_associations(kegg_associations_df)

# Validar resultados
print("\nNumero de asociaciones:", len(normalized_kegg_df))

print("Genes unicos:", normalized_kegg_df["locus_key"].nunique())
print("Rutas unicas:", normalized_kegg_df["kegg_pathway"].nunique())

display(normalized_kegg_df.head())

Cargando asociaciones KEGG existentes...

Numero de asociaciones: 5029
Genes unicos: 1823
Rutas unicas: 138


,kegg_gene,kegg_pathway,locus_key
0,pst:PSPTO_0083,pst00010,PSPTO0083
1,pst:PSPTO_0239,pst00010,PSPTO0239
2,pst:PSPTO_0387,pst00010,PSPTO0387
3,pst:PSPTO_0390,pst00010,PSPTO0390
4,pst:PSPTO_0728,pst00010,PSPTO0728


# Celda 2: Integración con resultados de InMoose
#### Identificar los DEGs que tienen anotación KEGG

In [3]:
# Definir la ruta de los resultados de expresion diferencial
deg_file = TABLES / "deg_results_completa.csv"

# Cargar los resultados de InMoose
expression_df = pd.read_csv(deg_file)

# Verificar que las columnas necesarias esten presentes
required_columns = {
    "locus_key",
    "log2FoldChange",
    "adj_pvalue"
}

missing_columns = required_columns - set(expression_df.columns)

if missing_columns:
    raise ValueError(f"Faltan columnas en los resultados: {missing_columns}")

# Integrar los resultados de expresion con las asociaciones KEGG
annotated_genes_df = expression_df.merge(
    normalized_kegg_df,
    on="locus_key",
    how="inner",
    validate="one_to_many"
)

# Identificar genes diferencialmente expresados
deg_mask = (
    (annotated_genes_df["adj_pvalue"] < 0.05)
    & (annotated_genes_df["log2FoldChange"].abs() > 1)
)

annotated_degs_df = annotated_genes_df.loc[deg_mask].copy()

# Mostrar un resumen
print("Genes evaluados en InMoose:", expression_df["locus_key"].nunique())
print("Genes evaluados con anotacion KEGG:", annotated_genes_df["locus_key"].nunique())
print("DEGs con anotacion KEGG:", annotated_degs_df["locus_key"].nunique())
print("Asociaciones gen-ruta de los DEGs:", len(annotated_degs_df))


display(annotated_degs_df.head())

Genes evaluados en InMoose: 5394
Genes evaluados con anotacion KEGG: 1735
DEGs con anotacion KEGG: 525
Asociaciones gen-ruta de los DEGs: 1523


,locus_key,baseMean,log2FoldChange,lfcSE,stat,pvalue,adj_pvalue,regulacion,gene,final_product,replicon,kegg_gene,kegg_pathway
0,PSPTO1232,12073.882594,6.31495,0.226863,27.835915,1.595180e-170,1.584652e-167,inducido,NaN,mannose-1-phosphate guanylyltransferase/mannos...,NC_004578.1,pst:PSPTO_1232,pst00051
1,PSPTO1232,12073.882594,6.31495,0.226863,27.835915,1.595180e-170,1.584652e-167,inducido,NaN,mannose-1-phosphate guanylyltransferase/mannos...,NC_004578.1,pst:PSPTO_1232,pst00520
2,PSPTO1232,12073.882594,6.31495,0.226863,27.835915,1.595180e-170,1.584652e-167,inducido,NaN,mannose-1-phosphate guanylyltransferase/mannos...,NC_004578.1,pst:PSPTO_1232,pst00541
3,PSPTO1232,12073.882594,6.31495,0.226863,27.835915,1.595180e-170,1.584652e-167,inducido,NaN,mannose-1-phosphate guanylyltransferase/mannos...,NC_004578.1,pst:PSPTO_1232,pst01100
4,PSPTO1232,12073.882594,6.31495,0.226863,27.835915,1.595180e-170,1.584652e-167,inducido,NaN,mannose-1-phosphate guanylyltransferase/mannos...,NC_004578.1,pst:PSPTO_1232,pst01110
